In [29]:
import joblib

PRODUCTS_DATA = joblib.load(r'C:\Users\khale\ML\RAG\Fashion RAG Assistant\data\clothes_json.joblib')

In [2]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\khale\anaconda3\envs\RAG\python.exe
3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]


In [7]:
%pip install arize-phoenix-otel opentelemetry-api

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from phoenix.otel import register

tracer_provider = register(
    endpoint="http://localhost:6006/v1/traces",
    project_name="fashion-rag-assistant"
)

OpenTelemetry Tracing Details
|  Phoenix Project: fashion-rag-assistant
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [ ]:
tracer = tracer_provider.get_tracer(__name__)


In [ ]:
from opentelemetry.trace import Status, StatusCode
from contextlib import contextmanager
from opentelemetry import trace
@contextmanager
def trace_span(span_name: str, attributes: dict = None):
    
    with tracer.start_as_current_span(span_name, attributes=attributes) as span:
        try:
            yield span
        except Exception as e:
            span.set_status(Status(StatusCode.ERROR, str(e)))
            span.record_exception(e)
            raise

In [30]:
PRODUCTS_DATA[0].keys()

dict_keys(['gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'price', 'product_id'])

In [31]:
PRODUCTS_DATA[0]

{'gender': 'Men',
 'masterCategory': 'Apparel',
 'subCategory': 'Topwear',
 'articleType': 'Shirts',
 'baseColour': 'Navy Blue',
 'season': 'Fall',
 'year': 2011.0,
 'usage': 'Casual',
 'productDisplayName': 'Turtle Check Men Navy Blue Shirt',
 'price': 67,
 'product_id': 15970}

In [32]:
FAQ = joblib.load(r"C:\Users\khale\ML\RAG\Fashion RAG Assistant\data\faq.joblib")

In [33]:
print(FAQ[0].keys())
FAQ[0]

dict_keys(['question', 'answer', 'type'])


{'question': 'What are your store hours?',
 'answer': 'Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday.',
 'type': 'general information'}

In [21]:
import ollama
import json

def check_if_faq_or_product(query: str) -> str:
    with tracer.start_as_current_span(
            "intent_classification",
            attributes={"input.value": query,
                        "openinference.span.kind": "LLM",
                        "llm.model_name": "deepseek-r1:7b"},
    
    ) as span:
        try:
            system_prompt = (
                'You are a strict text classification API. Classify the user\'s query into exactly one of these three classes: "FAQ", "Product", or "OTHER".\n\n'
                'You must output ONLY a single word representing the class and nothing else.\n\n'
                '- Use "FAQ" for policies, instructions, support, or general system questions.\n'
                '- Use "Product" for item availability, prices, recommendations, comparisons, or styling/outfit requests.\n'
                '- Use "OTHER" for greetings, general conversation, chit-chat, or topics unrelated to fashion products and company policies.'
            )

            messages = [
                {'role': 'system', 'content': system_prompt},
                # {'role': 'user', 'content': 'How do I reset my password?'},
                # {'role': 'assistant', 'content': '{"category": "FAQ"}'},
                # {'role': 'user', 'content': 'Do you have blue T-shirts under 100 dollars?'},
                # {'role': 'assistant', 'content': '{"category": "Product"}'},
                # {'role': 'user', 'content': 'what is the result of 5x + 5 = 25 '},
                # {'role': 'assistant', 'content': '{"category": "OTHER"}'},
                # {'role': 'user', 'content': 'how is the weather today'},
                # {'role': 'assistant', 'content': '{"category": "OTHER"}'},
                {'role': 'user', 'content': query},
            ]
                
            response = ollama.chat(
                model='deepseek-r1:7b',
                messages=messages,
                # think=False,       
                options={
                    'temperature': 0.0,
                    'num_predict': 700,   
                    
                }
            )
            prompt_tokens = response.get("prompt_eval_count", 0)
            completion_tokens = response.get("eval_count", 0)
            label = response.message.content.strip()
            label = label if label in ["FAQ", "Product", "OTHER"] else "OTHER"
            span.set_attribute("llm.token_count.prompt", prompt_tokens)
            span.set_attribute("llm.token_count.completion", completion_tokens)
            
            span.set_attribute("output.value", label)
            return label
        except Exception as e:
            span.set_attribute("output.value", "OTHER")
            span.set_status(Status(StatusCode.ERROR, str(e)))
            span.record_exception(e)
            return "OTHER"

In [22]:
queries = [
      'What is your return policy?', 
         # 'Give me three examples of blue T-shirts you have available.', 
         # 'How can I contact the user support?', 
         # 'Do you have blue Dresses?',
         # 'Create a look suitable for a wedding party happening during dawn.',
         # 'how are you ?',
         # 'Who won the basketball game yesterday',
         # 'I lost my password, how can I access my account',
         # 'Do you have anything to keep me warm in the winter?',
         # 'what is the result of 5x + 5 = 25',
         # 'I have a job interview tomorrow, what do you recommend I wear?'
]

for query in queries:
   response = check_if_faq_or_product(query)
   print(f"Query: {query} Label: {response}")

Query: What is your return policy? Label: FAQ


In [34]:
len(FAQ)

25

In [35]:
def generate_faq_layout(faq_dict: list) -> str:
    """
    Generates a formatted string layout for a list of FAQs.

    This function iterates through a dictionary of frequently asked questions (FAQs) and constructs
    a string where each question is followed by its corresponding answer and type.

    Parameters:
    - faq_dict (list): A list of dictionaries, each containing keys 'question', 'answer', and 'type' 
      representing an FAQ entry.

    Returns:
    - str: A string representing the formatted layout of FAQs, with each entry on a separate line.
    """
    text = " "
    for f in faq_dict:
        text += f'question: {f["question"]} Answer: {f["answer"]} Type: {f["type"]}\n '

    return text 

In [36]:
faq_layout = generate_faq_layout(FAQ)

In [37]:
print(len(faq_layout.split(" ")))
print(faq_layout[:1000])

724
 question: What are your store hours? Answer: Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday. Type: general information
 question: Where is Fashion Forward Hub located? Answer: Fashion Forward Hub is primarily an online store. Our corporate office is located at 123 Fashion Lane, Trend City, Style State. Type: general information
 question: Do you have a physical store location? Answer: At this time, we operate exclusively online. This allows us to offer a broader selection and lower prices directly to you. Type: general information
 question: How can I create an account with Fashion Forward Hub? Answer: Click on 'Sign Up' in the top right corner of our website and follow the instructions to set up your account. Type: general information
 question: How do I subscribe to your newsletter? Answer: To receive the latest updates and promotions, sign up for our newsletter at the bottom of our homepage. Type: general information


In [42]:
import ollama


def query_on_faq(query: str) -> str:
    with tracer.start_as_current_span(
            "faq_question_answering",
            attributes={"input.value": query,
                        "openinference.span.kind": "LLM",
                        "llm.model_name": "llama3.2"},
    
    ) as span:
        try:
            system_prompt = """
            You are a professional FAQ question-answering assistant.

            Your job is to answer the user's question using the FAQ context.

            """

            user_prompt = f"""
            Here is the FAQ context:

            {faq_layout}

            Here is the user's question:

            {query}


            Answer the user's question using only the FAQ context.
            """
            messages = [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]

            response = ollama.chat(
                model="llama3.2",
                messages=messages,
                options={
                    "temperature": 0.,
                    "num_predict": 200
                }
            )
            res = response.message.content.strip()
            span.set_attribute("output.value", res)
            return res
        except Exception as e:
            span.set_status(Status(StatusCode.ERROR, str(e)))
            span.record_exception(e)
            return "I'm sorry, I couldn't find an answer to your question in the FAQ."
        

In [46]:
faq_layout = generate_faq_layout(FAQ)
res = query_on_faq("I got my cloth but I didn't like it. How can I return it")

In [47]:
print(res)

To initiate a return, please follow these steps:

1. Go to our Returns Center and select the item you wish to exchange or return.
2. Choose the reason for return (if applicable).
3. Select the desired replacement or refund option.

Return processing typically takes 5-7 business days from when the item is received at our warehouse.


In [48]:
faq_layout

" question: What are your store hours? Answer: Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday. Type: general information\n question: Where is Fashion Forward Hub located? Answer: Fashion Forward Hub is primarily an online store. Our corporate office is located at 123 Fashion Lane, Trend City, Style State. Type: general information\n question: Do you have a physical store location? Answer: At this time, we operate exclusively online. This allows us to offer a broader selection and lower prices directly to you. Type: general information\n question: How can I create an account with Fashion Forward Hub? Answer: Click on 'Sign Up' in the top right corner of our website and follow the instructions to set up your account. Type: general information\n question: How do I subscribe to your newsletter? Answer: To receive the latest updates and promotions, sign up for our newsletter at the bottom of our homepage. Type: general information

In [120]:
@trace_span("decide_task_nature", {"openinference.span.kind": "LLM", "llm.model_name": "llama3.2"})
def decide_task_nature(query: str) -> str:

    current_span = trace.get_current_span()
    system_prompt = """Decide if the following query is a query that requires creativity (creating, composing, making new things) or technical (information about products, prices, etc.). 
    Label it as creative or technical.

    Examples:
    Query: Give me suggestions on a nice look for a nightclub.
    Label: creative

    Query: What are the blue dresses you have available?
    Label: technical

    Query: Give me three T-shirts for summer.
    Label: technical

    Query: Give me a look for attending a wedding party.
    Label: creative

    Only output one token: the label."""

    user_prompt = f"""
    question:
    
    {query}

    """
    messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
            
    response = ollama.chat(
        model="llama3.2",
        messages=messages,
        options={
            "temperature": 0.1,
            "num_predict": 200
        }
    )
    res = response.message.content.strip()
    
    current_span.set_attributes({
    "input.value": query,
    "llm.token_count.prompt": response.get("prompt_eval_count", 0),
    "llm.token_count.completion": response.get("eval_count", 0),
    "output.value": res,
    })
    current_span.set_status(Status(StatusCode.OK))
    return res

In [121]:
queries = ["Give me two sneakers with vibrant colors.",
           "What are the most expensive clothes you have in your catalogue?",
           "I have a green dress and I like a suggestion on an accessory to match with it.",
           "Give me three trousers with vibrant colors you have in your catalogue.",
           "Create a look for a woman walking in a park on a sunny day. It must be fresh due to hot weather."
           ]
for query in queries:
    label = decide_task_nature(query)
    print(f"Query: {query} Label: {label}")

Query: Give me two sneakers with vibrant colors. Label: technical
Query: What are the most expensive clothes you have in your catalogue? Label: technical
Query: I have a green dress and I like a suggestion on an accessory to match with it. Label: creative
Query: Give me three trousers with vibrant colors you have in your catalogue. Label: technical
Query: Create a look for a woman walking in a park on a sunny day. It must be fresh due to hot weather. Label: creative


In [124]:
def get_params_for_task(task: str) -> dict:
    
    PARAMETERS_DICT = {
        "creative": {"top_p": 0.7, 'temperature': 1.2},
        "technical": {'top_p': 0.9, 'temperature': 0.1}
    }

    return PARAMETERS_DICT.get(task, PARAMETERS_DICT["technical"])

In [125]:
get_params_for_task("technical")

{'top_p': 0.9, 'temperature': 0.1}

In [126]:
PRODUCTS_DATA[0]

{'gender': 'Men',
 'masterCategory': 'Apparel',
 'subCategory': 'Topwear',
 'articleType': 'Shirts',
 'baseColour': 'Navy Blue',
 'season': 'Fall',
 'year': 2011.0,
 'usage': 'Casual',
 'productDisplayName': 'Turtle Check Men Navy Blue Shirt',
 'price': 67,
 'product_id': 15970}

In [127]:
values = {}
for d in PRODUCTS_DATA:
    for key, val in d.items():
        if key in ('product_id', 'price', 'productDisplayName', 'subCategory', 'year'):
            continue
        if key not in values.keys():
            values[key] = set()
        values[key].add(val)

In [128]:
values['season']

{'All seasons', 'Fall', 'Spring', 'Summer', 'Winter'}

In [129]:
values['gender']

{'Boys', 'Girls', 'Men', 'Unisex', 'Women'}

In [130]:
@trace_span("generate_metadata_from_query", {"openinference.span.kind": "LLM", "llm.model_name": "llama3.2"})
def generate_metadata_from_query(query: str) -> str:
    system_prompt = f"""Given user query, your task is extract the following information and output in JSON format by strictly following given instructions.
    Information to extract:
    - **Gender**: Target audience for the product, such as "Men," "Women," or "Unisex."
    - **Master Category**: Broad classification like "Apparel" or "Footwear."
    - **Article Type**: Exact type of product, e.g., "Shirts" or "Jackets."
    - **Base Colour**: Main color of the product, important for customer choice.
    - **Season**: Intended season for the product, e.g., "Summer" or "Winter."
    - **Usage**: Intended use or occasion, like "Casual" or "Formal."
    - **Price**: Cost of the product.
    Instructions: 
    - Extract all information explained above
    - strictly use these keys names for JSON output: "gender", "masterCategory", "articleType", "baseColour", "price", "usage", "season".
    - values must be of list of string except 'price'. 'price' value must dictionary with two keys "min" and "max". If no 'price' mentioned, set "min" to 0 and "max" to "inf".
    - Only return Valid JSON output without anything else. 
    - here is example output json format.
    {{
        "gender": ["Women"],
        "masterCategory": ["Apparel"],
        "articleType": ["Dresses"],
        "baseColour": ["Blue"],
        "price": {{"min": 0, "max": "inf"}},
        "usage": ["Formal"],
        "season": ["All seasons"]
    }}
    
    """
    user_prompt = f"""
    query: 
    {query}
    """
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]
            
    response = ollama.chat(
        model="llama3.2",
        messages=messages,
        format= "json",
        options={
            "temperature": 0.1,
            "num_predict": 200
        }
    )
    res = response.message.content.strip()
    current_span = trace.get_current_span()
    current_span.set_attributes({
    "input.value": query,
    "llm.token_count.prompt": response.get("prompt_eval_count", 0),
    "llm.token_count.completion": response.get("eval_count", 0),
    "output.value": res,
    })
    current_span.set_status(Status(StatusCode.OK))
    return res

In [131]:
def parse_json_output(llm_output: str) -> dict:
    
    try:
        llm_output = llm_output.replace("\n", '').replace("'",'').replace("}}", "}").replace("{{", "{")  # Remove any erroneous structures
        
        parsed_json = json.loads(llm_output)
        return parsed_json
    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        return None

In [132]:
output = generate_metadata_from_query("Create a look for a man that suits a sunny day in the park. I don't want to spend more than 300 dollars on each piece." )

In [60]:
parse_json_output(output)

{'gender': ['Men'],
 'masterCategory': ['Apparel'],
 'articleType': ['Shirts', 'Shorts'],
 'baseColour': ['Navy Blue', 'Light Grey'],
 'price': {'min': 0, 'max': 300},
 'usage': ['Casual'],
 'season': ['Summer']}

In [133]:
json_string = generate_metadata_from_query("I need men shirt of 2xl, with price range 100 to 1000, for summer season in black color")
json_output = parse_json_output(json_string)
json_output

{'gender': ['Men'],
 'masterCategory': ['Apparel'],
 'articleType': ['Shirts'],
 'baseColour': ['Black'],
 'price': {'min': 100, 'max': 1000},
 'usage': ['Casual'],
 'season': ['Summer']}

In [62]:
json_string = generate_metadata_from_query("I need shirt for baby")
json_output = parse_json_output(json_string)
json_output

{'gender': ['Women', 'Men', 'Unisex'],
 'masterCategory': ['Apparel'],
 'articleType': ['Shirts'],
 'baseColour': ['White', 'Light Blue', 'Pastel Colors'],
 'price': {'min': 0, 'max': 'inf'},
 'usage': ['Formal', 'Casual', 'Baby'],
 'season': ['All seasons']}

In [134]:
print(len(PRODUCTS_DATA))

44424


In [135]:
PRODUCTS_DATA[0]

{'gender': 'Men',
 'masterCategory': 'Apparel',
 'subCategory': 'Topwear',
 'articleType': 'Shirts',
 'baseColour': 'Navy Blue',
 'season': 'Fall',
 'year': 2011.0,
 'usage': 'Casual',
 'productDisplayName': 'Turtle Check Men Navy Blue Shirt',
 'price': 67,
 'product_id': 15970}

In [136]:
def clean_record(record):

    cleaned = {}
    
    text_fields = ["gender", "masterCategory", "subCategory", "articleType", 
                   "usage", "season", "productDisplayName", "baseColour"]
    
    for field in text_fields:
        val = record[field]
        if val is None or str(val).strip() == "" or str(val).lower() == "nan":
            cleaned[field] = None
        else:
            cleaned[field] = str(val).strip()

    try:
        price_val = record.get("price")
        if price_val is None or str(price_val).strip() == "":
            cleaned["price"] = None
        else:
            cleaned["price"] = float(price_val)
    except ValueError:
        cleaned["price"] = None 

    try:
        id_val = record["product_id"]
        if id_val is None or str(id_val).strip() == "":
            cleaned["product_id"] = None
        else:
            cleaned["product_id"] = int(id_val)
    except ValueError:
        cleaned["product_id"] = None

    return cleaned

In [84]:
import weaviate

client = weaviate.connect_to_local(port=8090, grpc_port=50051)

client.is_ready()

True

In [86]:
from weaviate.classes.config import Configure, Property, DataType

if not client.collections.exists("products"):
    collections = client.collections.create(
        name= "products",

        vectorizer_config=Configure.Vectorizer.text2vec_ollama(
            model="nomic-embed-text",
            api_endpoint="http://host.docker.internal:11434",
            vectorize_collection_name=False
             
        ),
            properties = [
            Property(name="productDisplayName", data_type=DataType.TEXT),
            Property(name="articleType", data_type=DataType.TEXT),
            Property(name="baseColour", data_type=DataType.TEXT),
            Property(name="usage", data_type=DataType.TEXT),
            Property(name="season", data_type=DataType.TEXT),
            
            Property(name="price", data_type=DataType.NUMBER, skip_vectorization=True),
            Property(name="product_id", data_type=DataType.INT, skip_vectorization=True),
            Property(name="year", data_type=DataType.TEXT, skip_vectorization=True),
            Property(name="gender", data_type=DataType.TEXT, skip_vectorization=True) 
        ]
    )
else : 
    collections = client.collections.get("products")

c:\Users\khale\anaconda3\envs\RAG\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


In [89]:
import tqdm
collection = client.collections.get("products")

with collection.batch.dynamic() as batch:
    for record in tqdm.tqdm(PRODUCTS_DATA):
        cleaned_properties = clean_record(record)
        
        batch.add_object(
            properties=cleaned_properties
        )
        
if len(collection.batch.failed_objects) > 0:
    print(f"{len(collection.batch.failed_objects)}")
    print(collection.batch.failed_objects[0].message)
else:
    print("done ")

c:\Users\khale\anaconda3\envs\RAG\Lib\site-packages\weaviate\warnings.py:312: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\khale\AppData\Local\Temp\ipykernel_33948\765811168.py:2: ResourceWarning: unclosed <socket.socket fd=2536, family=23, type=1, proto=0, laddr=('::1', 62815, 0, 0), raddr=('::1', 8090, 0, 0)>
  collection = client.collections.get("products")
  0%|          | 0/44424 [00:00<?, ?it/s]

 16%|█▌        | 7056/44424 [00:59<05:59, 103.81it/s]c:\Users\khale\anaconda3\envs\RAG\Lib\site-packages\google\protobuf\internal\well_known_types.py:616: ResourceWarning: unclosed <socket.socket fd=1780, family=23, type=1, proto=0, laddr=('::1', 55404, 0, 0), raddr=('::1', 8090, 0, 0)>
  for key, value in dictionary.items():
100%|██████████| 44424/44424 [07:28<00:00, 98.96it/s] 


done 


In [90]:
print(len(client.collections.get("products")))


44424


In [91]:
from weaviate.classes.query import Filter
def get_filter_by_metadata(json_output: dict | None = None):

    if json_output == None:
        return None
    
    valid_keys = (
        'gender',
        'masterCategory',
        'articleType',
        'baseColour',
        'price',
        'usage',
        'season',
    )

    filters = []
    for key, value in json_output.items():

        if key not in valid_keys:
            continue

        if key == "price":

            if not isinstance(value, dict):
                continue

            max_price = value.get('max')
            min_price = value.get('min')

            if max_price == None or min_price == None:
                continue

            if min_price > 0:
                filters.append(Filter.by_property(key).greater_or_equal(min_price))
            
            if max_price != 'inf':
                filters.append(Filter.by_property(key).less_or_equal(max_price))

        else:
            filters.append(Filter.by_property(key).contains_any(value))

    return filters

In [137]:
def generate_filters_from_query(query: str) -> list:
    json_string = generate_metadata_from_query(query)
    json_output = parse_json_output(json_string)
    filters = get_filter_by_metadata(json_output)
    return filters

In [138]:
filters = generate_filters_from_query("Give me three T-shirts to use in sunny days")
filters 


[_FilterValue(value=['Men', 'Women'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='gender'),
 _FilterValue(value=['Apparel'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='masterCategory'),
 _FilterValue(value=['T-Shirts'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='articleType'),
 _FilterValue(value=['White', 'Light Blue', 'Yellow'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='baseColour'),
 _FilterValue(value=['Casual'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='usage'),
 _FilterValue(value=['Summer'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='season')]

In [59]:
filters[-1].target

'season'

In [139]:
from weaviate.classes.query import Filter
@trace_span("get_relevant_products_from_query", {"openinference.span.kind": "LLM", "llm.model_name": "nomic-embed-text"})
def get_relevant_products_from_query(query: str):
    filters = generate_filters_from_query(query)

    if filters is None or len(filters) == 0:
        response = collections.query.near_text(
            query=query,
            limit=20,
        ).objects 
        return response
    
    response = collections.query.near_text(
        query=query,
        limit=20,
        filters=Filter.all_of(filters)
    ).objects

    if len(response) >= 10:
        return response

    importance_order = [
        'gender',          
        'masterCategory',  
        'articleType',     
        'price',           
        'baseColour',     
        'season',        
        'usage',      
        'year'        
    ]
    
    for i in range(1, len(importance_order)):
        
        keys_to_keep = importance_order[:-i] 

        current_filters = [f for f in filters if f.target in keys_to_keep]

        if len(current_filters) == 0:
            break

        response = collections.query.near_text(
            query=query,
            limit=20,
            filters=Filter.all_of(current_filters) 
        ).objects
        
        if len(response) >= 5:
            return response

    response = collections.query.near_text(
        query=query,
        limit=20,
    ).objects
    current_span = trace.get_current_span()
    current_span.set_attributes({
        "input.value": query,
        "llm.token_count.prompt": 0,
        "llm.token_count.completion": 0,
        "output.value": f"Returned {len(response)} products without filters"
    })  
    
    return response

In [140]:
test = get_relevant_products_from_query("Give me three T-shirts to use in sunny days")

In [96]:
for i in test:
    print(i.properties)

{'gender': 'Men', 'price': 250.0, 'usage': 'Casual', 'subCategory': 'Topwear', 'baseColour': 'Beige', 'season': 'Summer', 'product_id': 37152, 'year': None, 'articleType': 'Tshirts', 'masterCategory': 'Apparel', 'productDisplayName': 'Campbell Men Pack of 3 T-shirts'}
{'gender': 'Men', 'price': 97.0, 'masterCategory': 'Apparel', 'subCategory': 'Topwear', 'baseColour': 'Blue', 'season': 'Summer', 'product_id': 24818, 'year': None, 'articleType': 'Tshirts', 'usage': 'Casual', 'productDisplayName': 'Basics Men Pack of 3 T-shirts'}
{'gender': 'Men', 'price': 253.0, 'usage': 'Casual', 'articleType': 'Tshirts', 'baseColour': 'Pink', 'season': 'Summer', 'product_id': 12225, 'year': None, 'masterCategory': 'Apparel', 'subCategory': 'Topwear', 'productDisplayName': 'Basics Men Pack of 3 T-shirts'}
{'gender': 'Men', 'price': 142.0, 'masterCategory': 'Apparel', 'subCategory': 'Topwear', 'baseColour': 'Beige', 'season': 'Summer', 'product_id': 37154, 'year': None, 'usage': 'Casual', 'articleType':

In [141]:
def generate_items_context(results: list) -> str:
    res = ""
    for obj in results:
        properties = obj.properties
        res += (
            f"Product ID: {properties.get('product_id')}, "
            f"Product: {properties.get('productDisplayName')}, "
            f"Price: {properties.get('price')}, "
            f"Color: {properties.get('baseColour')}, "
            f"Category: {properties.get('articleType')}, "
            f"Gender: {properties.get('gender')}, "
            f"Season: {properties.get('season')}.\n"
            
        )
    return res

In [142]:
print(generate_items_context(test)[:1000])

Product ID: 37152, Product: Campbell Men Pack of 3 T-shirts, Price: 250.0, Color: Beige, Category: Tshirts, Gender: Men, Season: Summer.
Product ID: 24818, Product: Basics Men Pack of 3 T-shirts, Price: 97.0, Color: Blue, Category: Tshirts, Gender: Men, Season: Summer.
Product ID: 12225, Product: Basics Men Pack of 3 T-shirts, Price: 253.0, Color: Pink, Category: Tshirts, Gender: Men, Season: Summer.
Product ID: 37154, Product: Campbell Men Pack of 3 T-shirts, Price: 142.0, Color: Beige, Category: Tshirts, Gender: Men, Season: Summer.
Product ID: 20284, Product: Campbell Men Pack of 3 T-shirts, Price: 150.0, Color: Green, Category: Tshirts, Gender: Men, Season: Summer.
Product ID: 47282, Product: Myntra Men Pack of 3 T-shirts, Price: 125.0, Color: Pink, Category: Tshirts, Gender: Men, Season: Summer.
Product ID: 7891, Product: Proline Men Pack of 3 T-shirts, Price: 172.0, Color: Cream, Category: Tshirts, Gender: Men, Season: Fall.
Product ID: 47280, Product: Myntra Men Pack of 3 T-shir

In [143]:
import ollama
@trace_span("query_on_products", {"openinference.span.kind": "LLM", "llm.model_name": "llama3.2"})
def query_on_products(query: str) -> str: 
    query_label = decide_task_nature(query)
    parameters_dict = get_params_for_task(query_label)
    relevant_products = get_relevant_products_from_query(query)
    context = generate_items_context(relevant_products)

    system_prompt = (
        "Given the available set of cloth products, answer the question that follows, providing the item ID in your answers. "
        "Other information might be provided but not necessarily all of them; pick only the relevant ones for the given query and avoid being too long when describing the items' features. "
        "If no number of products is mentioned in the query, select at most five to show. "
        "Act as a helpful fashion assistant."
    )

    user_prompt = (
        f"CLOTH PRODUCTS AVAILABLE: \n{context}\n\n"
        f"QUERY: {query}"
    )
    
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]
    
    response = ollama.chat(
        model="llama3.2",
        messages=messages,
        options={
            "temperature": parameters_dict.get('temperature', 0.7),
            "top_p": parameters_dict.get('top_p', 0.9),
            "num_predict": 300 
        }
    )
    current_span = trace.get_current_span()
    current_span.set_attributes({
        "input.value": query,
        "llm.token_count.prompt": response.get("prompt_eval_count", 0),
        "llm.token_count.completion": response.get("eval_count", 0),
        "output.value": response['message']['content'].strip()
    })

    return response['message']['content'].strip()

In [146]:
import ollama

chat_memory = []
@trace_span("answer_query", {"openinference.span.kind": "LLM", "llm.model_name": "llama3.2"})
def answer_query(query: str):
    global chat_memory 
    current_span = trace.get_current_span()
    label = check_if_faq_or_product(query)
    final_response = "" 

    if label == "FAQ":
        final_response = query_on_faq(query)

    elif label == "Product":
        try:
            final_response = query_on_products(query)
        except Exception as e:
            system_prompt = (
                "User provided a question that broke the querying system. Instruct them to rephrase it. "
                "Answer it based on the context you already have so far."
            )
            user_prompt = f"User question {query}"

            messages = [{'role': 'system', 'content': system_prompt}]
            messages.extend(chat_memory) 
            messages.append({'role': 'user', 'content': user_prompt})

            response = ollama.chat(
                model="llama3.2",
                messages=messages,
                options={
                    "temperature": 0.7,
                    "num_predict": 300 
                }
            )
            final_response = response['message']['content'].strip() 
            current_span = trace.get_current_span()
            current_span.set_attributes({
                "input.value": query,
                "llm.token_count.prompt": response.get("prompt_eval_count", 0),
                "llm.token_count.completion": response.get("eval_count", 0),
                "output.value": final_response
            })

    else:
        system_prompt = (
            "You are a helpful assistant. The user provided a question that does not fit FAQ or Product related questions. "
            "Answer it based on the context you already have so far."
        )
        user_prompt = f"user question {query}"

        messages = [{'role': 'system', 'content': system_prompt}]
        messages.extend(chat_memory)
        messages.append({'role': 'user', 'content': user_prompt})
            
        response = ollama.chat(
            model="llama3.2",
            messages=messages,
            options={
                "temperature": 0.7,
                "num_predict": 300 
            }
        )
        final_response = response['message']['content'].strip()
        current_span.set_attributes({
            "input.value": query,
            "llm.token_count.prompt": response.get("prompt_eval_count", 0),
            "llm.token_count.completion": response.get("eval_count", 0),
            "output.value": final_response
        }) 
    
    chat_memory.append({'role': 'user', 'content': query})
    chat_memory.append({'role': 'assistant', 'content': final_response})
    
    if len(chat_memory) > 6:
        chat_memory = chat_memory[-6:]

    
    current_span = trace.get_current_span()
    current_span.set_attributes({
        "input.value": query,
        "output.value": final_response
    })
    return final_response

In [147]:
test1 = answer_query("What are your working hours?")
# test2 = answer_query("Tomorrow is my engagement day Suggest what to wear")

In [102]:
print(test1)
print("-" * 50)
# print(test2)

Our customer service is available from 9:00 AM to 6:00 PM, Monday through Friday. However, our online store is open 24/7.
--------------------------------------------------
